# Calcul des performances
- Évaluation  

## Importations
- codecs pour les encodages
- pandas et numpy pour les calculs sur tableaux
- matplotlib pour les graphiques
- itertools pour les itérateurs sophistiqués (paires sur liste, ...)

In [1]:
# -*- coding: utf8 -*-
# import codecs,operator,datetime,os,glob
# import features
import re,codecs
import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import itertools as it
import yaml
# import networkx as nx
#%pylab inline
#pd.options.display.mpl_style = 'default'
# debug=False
# from __future__ import print_function

def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [2]:
import yaml

In [3]:
from IPython.display import display, HTML

In [4]:
import datetime
def dateheure():
    return datetime.datetime.utcnow().strftime('%y%m%d%H%M')

In [5]:
saut="\n"

# Choix de l'échantillon et du gold
- *inputType* est le type de l'échantillon de départ
    - CVk-Type pour les k-fold (k le nombre de morceaux)
        - Type=Train pour le k-fold standard
        - Type=Test pour le k-fold inverse
    - S pour les déciles, 8 pour les 10%, 7 pour les 20%, 6 pour les 30%, 5 pour les 40%, 4 pour les 50%
- *checkType* est le type de contrôle pour les formes
    - gold pour un contrôle correspondant aux formes attestées
    - platinum pour un contrôle correspondant à l'ensemble théorique des formes
- *num* est le numéro de l'input considéré

In [29]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
inputType="CV5-Test"
inputType="S"
checkType="platinum"
# checkType="gold"
num=7

if inputType=="S":
    inputFile="vlexique2-S%d.csv"%num
    outputFile="vlexique2-S%d-omp-Swim2.csv"%num
    goldFile="vlexique2-R%d.csv"%num
elif "CV" in inputType:
    quant,known=inputType.split("-")
    if known=="Train":
        predict="Test"
    elif known=="Test":
        predict="Train"
    inputFile="vlexique2-%s-%s%d.csv"%(quant,known,num)
    outputFile="vlexique2-%s-%s%d-omp-Swim2.csv"%(quant,known,num)
    goldFile="vlexique2-%s-%s%d.csv"%(quant,predict,num)
    
platinumFile="vlexique2-Total.csv"
statistiquesFile="vlexique2-Statistiques.yaml"

fInput=repFiles+inputFile
fOutput=repFiles+outputFile
fGold=repFiles+goldFile
fPlatinum=repFiles+platinumFile
fStatistiques=repFiles+statistiquesFile

In [7]:
phonologicalMap="-X"
if "omp" in outputFile:
    casesType="-Morphomes"
else:
    casesType=""
listeFormesOutput=["FS","FP"]

### Dédoubler les lignes avec des surabondances dans *colonne*
>identifier une ligne avec surabondance

>>ajouter les lignes correspondant à chaque valeur

>>ajouter le numéro de la ligne initiale dans les lignes à supprimer

>supprimer les lignes avec surabondance

NB : il faut préparer le tableau pour avoir une indexation qui permette l'ajout des valeurs individuelles et la suppression des lignes de surabondances

# Lecture de l'échantillon

In [8]:
neutralisationsNORD=(u"6û",u"9ê")
neutralisationsSUD=(u"e2o",u"E9O")
if phonologicalMap=="-N":
    neutralisations=neutralisationsNORD
elif phonologicalMap=="-S":
    neutralisations=neutralisationsSUD
else:
    neutralisations=(u"",u"")
    phonologicalMap=("-X")
bdlexiqueIn = u"èò"+neutralisations[0]
bdlexiqueNum = [ord(char) for char in bdlexiqueIn]
neutreOut = u"EO"+neutralisations[1]
neutralise = dict(zip(bdlexiqueNum, neutreOut))

neutralisationsTotales=(u"e2o6û",u"E9O9ê")
totalNeutreIn=u"èò"+neutralisationsTotales[0]
totalNeutreNum=[ord(char) for char in totalNeutreIn]
totalNeutreOut=u"EO"+neutralisationsTotales[1]
totalNeutralise = dict(zip(totalNeutreNum, totalNeutreOut))

In [9]:
def recoder(chaine,table=totalNeutralise):
    if type(chaine)==str:
        temp=chaine.translate(table)
        result=temp
    elif type(chaine)==unicode:
        result=chaine.translate(table)
    else:
        result=chaine
    return result

### Vérification de la phonotactique des glides du français
- si *prononciation* est *None* renvoyer *None*
- ajout de diérèses dans les séquences mal-formées
- vérification des séquences consonne+glide à la finale

In [11]:
dierese={"j":"ij", "w":"uw","H":"yH","i":"ij","u":"uw","y":"yH"}
glide2voc={"j":"i","w":"u","H":"y"}

In [12]:
def checkFrench(prononciation):
    if prononciation and not pd.isnull(prononciation):
        result=recoder(prononciation)
        # Consonne plus glide final
        m=re.match(r"^(.*[^ieèEaOouy926êôâ])([jwH])$",result)
        if m:
            # print ("pb avec un glide final", [prononciation])
            result=m.group(1)+glide2voc[m.group(2)]
        # attaque Obs+Liq+Glide
        m=re.match(r"(.*[ptkbdgfsSvzZ][rl])([jwH])(.*)",result)
        if m:
            n=re.search(r"[ptkbdgfsSvzZ][rl](wa|Hi|wê)",result)
            if not n:
                glide=m.group(2)
                result=m.group(1)+dierese[glide]+m.group(3)
        # Voyelle haute+Voyelle => diérèse
        m=re.match(r"(.*)([iuy])([ieEaOouy].*)",result)
        if m:
            glide=m.group(2)
            result=m.group(1)+dierese[glide]+m.group(3)
        # yod ou n palatal+yod
        m=re.match(r"(.*)([jJ])(j)(.*)",result)
        if m:
            result=m.group(1)+m.group(2)+m.group(4)
            # print(prononciation,"=>",result)
        m=re.match(r"^(.*[^ieèEaOouy926êôâ])([jwH])6(.*)$",result)
        if m:
            result=m.group(1)+glide2voc[m.group(2)]+m.group(3)
            # print(prononciation,"=>",result)
    else:
        result=prononciation
    return result

In [14]:
def flattenData(tData):
    dfData=tData.melt("lexeme",var_name="cell",value_name="form")
    dfData=dfData.dropna(thresh=3).set_index(["lexeme", "cell"]).apply(lambda x: x.str.split(',').explode()).sort_values("lexeme").reset_index()
    return dfData
    
def separateData(df1,df2):
    df1=pd.merge(df1,df2,on=["lexeme","cell","form"],how="left",indicator=True)
    df1=df1.loc[df1["_merge"]=="left_only"].drop("_merge", axis=1)
    return df1

In [15]:
tInput=pd.read_csv(fInput,sep=";",encoding="utf8")
if u"Unnamed: 0" in tInput.columns:
    del tInput[u"Unnamed: 0"]
tInput=tInput.dropna(axis=1,how='all')
print(len(tInput.columns))
dfInput=flattenData(tInput)
dfInput.form=dfInput.form.apply(lambda x: checkFrench(x))
dfInput

47


,lexeme,cell,form
0,abaisser,inf,abEsE
1,abaisser,ppMS,abEsE
2,abaisser,pI2S,abEs
3,abaisser,fi1S,abEs9rE
4,abaisser,pI2P,abEsE
...,...,...,...
21753,être,pc1S,s9rE
21754,être,pc1P,s9rjô
21755,être,pP,Etâ
21756,être,pI2P,swajE


le tableau tOutput contient la sortie de SWIM avec toutes les formes (initiales et générées) sous forme de paradigmes  
- les surabondances sont dans la même case séparées par une virgule dans tOutput

le tableau dfOutput contient seulement les formes générées, une par ligne  
- les surabondances sont sur deux lignes différentes dans dfOutput

In [16]:
tOutput=pd.read_csv(fOutput,sep=";",encoding="utf8")
if u"Unnamed: 0" in tOutput.columns:
    del tOutput[u"Unnamed: 0"]
tOutput=tOutput.dropna(axis=1,how='all')
# print((tOutput.columns))
dfOutput=flattenData(tOutput)
dfOutput.form=dfOutput.form.apply(lambda x: checkFrench(x))
dfOutput=separateData(dfOutput,dfInput).reset_index().drop("index", axis=1)
dfOutput

,lexeme,cell,form
0,abaisser,ai3S,abEsa
1,abaisser,pi2S,abEs
2,abaisser,fi1P,abEs9rô
3,abaisser,fi3S,abEs9ra
4,abaisser,pc2S,abEs9rE
...,...,...,...
67248,évoquer,pc3P,EvOk9rE
67249,évoquer,ps3S,EvOk
67250,évoquer,ppFP,EvOkE
67251,évoquer,ps2P,EvOkjE


In [17]:
sampleCases=tInput.columns.values.tolist()
sampleCases.remove(u"lexeme")
# sampleCases
analyseCases=sampleCases

#Adapt all the forms to French phonology
for case in sampleCases:
    tInput[case]=tInput[case].apply(lambda x: checkFrench(x))

tInput.loc[tInput.lexeme=="affluer"].T.dropna()

,81
lexeme,affluer
inf,aflyHE
pi3P,afly
pi3S,afly


- sampleCases pour la liste des cases effectivement représentées dans le corpus de départ 

In [18]:
countInput=tInput[sampleCases].stack().value_counts(dropna=True).sum()
print("nombre de formes de départ",countInput)

nombre de formes de départ 21735


In [30]:
if checkType=="gold":
    nGold=fGold
elif checkType=="platinum":
    nGold=fPlatinum

tGold=pd.read_csv(nGold,sep=";",encoding="utf8")
if u"Unnamed: 0" in tGold.columns:
    del tGold[u"Unnamed: 0"]
tGold=tGold.dropna(axis=1,how='all')
for case in sampleCases:
    tGold[case]=tGold[case].apply(lambda x: checkFrench(x))


dfGold=flattenData(tGold)
dfGold.form=dfGold.form.apply(lambda x: checkFrench(x))
dfGold=separateData(dfGold,dfInput).reset_index().drop("index", axis=1)
dfGold

,lexeme,cell,form
0,abaisser,ai1P,abEsam
1,abaisser,ii1P,abEsjô
2,abaisser,ppFP,abEsE
3,abaisser,fi3S,abEs9ra
4,abaisser,fi3P,abEs9rô
...,...,...,...
246738,être,ai2P,fyt
246739,être,is2S,fys
246740,être,is3P,fys
246741,être,ppMP,EtE


In [20]:
countPredictions=dfGold.loc[dfGold.cell.isin(sampleCases)].value_counts(dropna=True).sum()
print("nombre de formes à générer",countPredictions)

nombre de formes à générer 87088


In [21]:
tPlatinum=pd.read_csv(fPlatinum,sep=";",encoding="utf8")
if u"Unnamed: 0" in tPlatinum.columns:
    del tPlatinum[u"Unnamed: 0"]
tPlatinum=tPlatinum.dropna(axis=1,how='all')

for case in sampleCases:
    tPlatinum[case]=tPlatinum[case].apply(lambda x: checkFrench(x))

dfPlatinum=flattenData(tPlatinum)
dfPlatinum.form=dfPlatinum.form.apply(lambda x: checkFrench(x))
dfPlatinum

,lexeme,cell,form
0,abaisser,ai1P,abEsam
1,abaisser,pi3S,abEs
2,abaisser,ii1P,abEsjô
3,abaisser,ppFP,abEsE
4,abaisser,fi3S,abEs9ra
...,...,...,...
268496,être,fi3S,s9ra
268497,être,ppMP,EtE
268498,être,ai2S,fy
268499,être,is1S,fys


In [22]:
dfPlatinum.loc[dfPlatinum.cell=="fi1S"]

,lexeme,cell,form
11,abaisser,fi1S,abEs9rE
59,abandonner,fi1S,abâdOn9rE
148,abasourdir,fi1S,abazurdirE
182,abattre,fi1S,abatrE
231,abdiquer,fi1S,abdik9rE
...,...,...,...
268257,éviscérer,fi1S,EvisEr9rE
268313,éviter,fi1S,Evit9rE
268348,évoluer,fi1S,EvOlyrE
268439,évoquer,fi1S,EvOk9rE


# Identifier les lexèmes potentiellement générés

In [23]:
%%time
testLexemes=[]
outLexemes={}
for ix,row in tInput.iloc[:,:].iterrows():
    if row.lexeme in testLexemes:display(row.dropna())
    if ix%100==0: print(ix)
    dCriteres=row.dropna().to_dict()
    del dCriteres["lexeme"]
    if row.lexeme in testLexemes:print(dCriteres)
    selCriteres=[]
    for k,v in dCriteres.items():
        if "," in v:
            lVs=v.split(",")
            for lV in lVs:
                selCriteres.append("((dfPlatinum.cell=='%s') & (dfPlatinum.form=='%s'))"%(k,lV))
        else:
            selCriteres.append("((dfPlatinum.cell=='%s') & (dfPlatinum.form=='%s'))"%(k,v))
    nbInput=len(selCriteres)
    exec("%s=%s"%("testPlatinum","|".join(selCriteres)))
    if row.lexeme in testLexemes:
        print("|".join(selCriteres))
        display(dfPlatinum.loc[testPlatinum])
    lLexemes=dfPlatinum.loc[testPlatinum].lexeme.unique().tolist()
    notLexemes=[]
    for lexeme in lLexemes:
        nbPlatinum=len(dfPlatinum.loc[(testPlatinum)&(dfPlatinum.lexeme==lexeme)])
        if row.lexeme in testLexemes:
            display(dfPlatinum.loc[dfPlatinum.lexeme==lexeme])
            print(lexeme,nbPlatinum)
        if nbPlatinum<nbInput:
            notLexemes.append(lexeme)
            # print(row.lexeme,lexeme,nbInput,nbPlatinum)
    lLexemes=[l for l in lLexemes if l not in notLexemes]
    outLexemes[row.lexeme]=lLexemes
    if len(lLexemes)>1:
        print("plusieurs candidats",lLexemes)
        print(row.dropna())
    elif len(lLexemes)==0:
        print("pas de candidat",row.dropna())

0
100
plusieurs candidats ['ailler', 'aller']
lexeme    ailler
ps1S          aj
ps3P          aj
ps3S          aj
Name: 102, dtype: object
200
300
400
500
600
plusieurs candidats ['desserrer', 'desservir']
lexeme    desservir
pi3S          dEsEr
Name: 680, dtype: object
700
800
900
plusieurs candidats ['dépourvoir', 'pourvoir']
lexeme    dépourvoir
ppFS           purvy
ppMP           purvy
ppMS           purvy
Name: 922, dtype: object
1000
1100
1200
1300
1400
plusieurs candidats ['enter', 'hanter']
lexeme    hanter
fi3S       ât9ra
ii3S         âtE
inf          âtE
pi3P          ât
pi3S          ât
ppFP         âtE
ppFS         âtE
ppMP         âtE
ppMS         âtE
Name: 1403, dtype: object
1500
1600
plusieurs candidats ['luncher', 'lyncher']
lexeme    lyncher
inf          lêSE
ppMS         lêSE
Name: 1608, dtype: object
1700
1800
1900
plusieurs candidats ['dépourvoir', 'pourvoir']
lexeme    pourvoir
inf        purvwar
ppMS         purvy
Name: 1933, dtype: object
2000
2100
plusieurs ca

In [24]:
# outLexemes

# Calcul des performances

In [31]:
def assembleTriplet(triplet):
    result={}
    for k,v in triplet.items():
        result[k[1]]=v
    return result
    

def compareGold(predLexeme,goldLexeme):
    positive={}
    negative={}
    missing={}
    
    (_,goldCells),(_,goldForms)=dfGold.loc[dfGold.lexeme==goldLexeme][["cell","form"]].sort_values("cell").to_dict().items()
    (_,predCells),(_,predForms)=dfOutput.loc[dfOutput.lexeme==predLexeme][["cell","form"]].sort_values("cell").to_dict().items()
    
    for k,goldCell in goldCells.items():
        for kk,predCell in predCells.items():
            if predCell==goldCell and predForms[kk]==goldForms[k]:
                positive[(k,goldCell)]=goldForms[k]
                break
                
    for k,goldCell in goldCells.items():
        if (k,goldCell) not in positive:
            for kk,predCell in predCells.items():
                if predCell==goldCell:
                    negative[(k,goldCell)]=predForms[kk]+"≠"+goldForms[k]
                    break
                    
    for k,goldCell in goldCells.items():
        if (k,goldCell) not in positive and (k,goldCell) not in negative:
            missing[(k,goldCell)]="Ø≠"+goldForms[k]
    return assembleTriplet(positive),assembleTriplet(negative),assembleTriplet(missing)


In [32]:
%%time
connu={}
correct={}
different={}
missing={}
for predLexeme,checks in dict(list(outLexemes.items())[:]).items():
    if len(checks)>1: print(predLexeme,checks)
    connu[predLexeme]=dfInput.loc[dfInput.lexeme==predLexeme].set_index("cell")["form"].to_dict()
    maxPositive=0
    minNegative=len(sampleCases)
    for goldLexeme in checks:
        lPositive,lNegative,lMissing=compareGold(predLexeme,goldLexeme)
        if len(lPositive)>maxPositive:
            maxPositive=len(lPositive)
            correct[predLexeme]=lPositive
            different[predLexeme]=lNegative
            missing[predLexeme]=lMissing
        elif len(lPositive)==maxPositive:
            if len(lNegative)<minNegative:
                minNegative=len(lNegative)
                correct[predLexeme]=lPositive
                different[predLexeme]=lNegative
                missing[predLexeme]=lMissing


ailler ['ailler', 'aller']
desservir ['desserrer', 'desservir']
dépourvoir ['dépourvoir', 'pourvoir']
hanter ['enter', 'hanter']
lyncher ['luncher', 'lyncher']
pourvoir ['dépourvoir', 'pourvoir']
receler ['receler', 'recéler']
égayer ['égailler', 'égayer']
CPU times: user 29.3 s, sys: 87.4 ms, total: 29.4 s
Wall time: 29.4 s


In [33]:
def countForms(dForms):
    df=pd.DataFrame.from_dict(dForms)
    nb=df[df.notnull()].count().sum()
    return int(nb)
    
nbConnu=countForms(connu)
nbCorrect=countForms(correct)
nbDifferent=countForms(different)
nbMissing=countForms(missing)

precision=float(nbCorrect)/(nbCorrect+nbDifferent)*100
rappel=float(nbCorrect)/(nbCorrect+nbMissing)*100
print("input :",inputFile,"output :",outputFile,"check type :",checkType)
print("Brut précision %.1f, rappel %.1f"%(precision,rappel))


input : vlexique2-S7.csv output : vlexique2-S7-omp-Swim2.csv check type : platinum
Brut précision 98.7, rappel 52.3


In [34]:
def yamlDump(nFile,content):
    with open(nFile, 'w') as output:
        yaml.dump(content, output, default_flow_style=False,allow_unicode=True)

    with open(nFile, 'r') as input:
        yamlLines=input.readlines()

    yamlText="".join(yamlLines)
    yamlText=re.sub(r"!!python/unicode","",yamlText)
    yamlText=re.sub(r"\n\s*-\s*",", ",yamlText)
    yamlText=re.sub(r":,\s*",": ",yamlText)

    with open(nFile, 'w') as output:
        output.write(yamlText)
    return


stats={}
with open(fStatistiques,"r") as inFile:
     stats=yaml.safe_load(inFile)
if not stats:
    stats={}
if inputType+str(num) not in stats:
    stats[inputType+str(num)]={}
stats[inputType+str(num)][checkType]={}
stats[inputType+str(num)][checkType]["nbConnu"]=nbConnu
stats[inputType+str(num)][checkType]["nbCorrect"]=nbCorrect
stats[inputType+str(num)][checkType]["nbDifferent"]=nbDifferent
stats[inputType+str(num)][checkType]["nbMissing"]=nbMissing
stats[inputType+str(num)][checkType]["precision"]=precision
stats[inputType+str(num)][checkType]["rappel"]=rappel

yamlDump(fStatistiques,stats)